# Anomaly calculations

In [1]:
from earthkit import data as ekd
from earthkit import plots as ekp
from earthkit import transforms as ekt
from earthkit.transforms._tools import earthkit_remote_test_data_file

## Load some test data

In this example we will use hourly ERA5 2m temperature data on a 3˚ X 3˚ spatial grid for the years 2015 to 2017 as
our physical data.

All `earthkit-transforms` methods can be called with `earthkit-data` objects (Readers and Wrappers)
or with a pre-loaded `xarray`.
To reduce the number of conversions in the example, we will convert to xarray in the first cell
and use that data object for all subsequent steps. This also allows us to set some dataset specific options
in the conversion to xarray instead of relying on the default options.

In [2]:
# Get some demonstration ERA5 data, this could be any url or path to an ERA5 grib or netCDF file.
remote_era5_file = earthkit_remote_test_data_file("era5-Europe-sfc-2m-temperature-3deg-2015-2017.grib")
era5_data = ekd.from_source("url", remote_era5_file)
# convert to xarray to save repeated conversion in further steps.
# the .compute loads everything into memory which is safe for this example
era5_xr = era5_data.to_xarray(time_dim_mode="valid_time").compute()
era5_xr

<xarray.Dataset> Size: 19MB
Dimensions:     (valid_time: 4384, latitude: 19, longitude: 29)
Coordinates:
  * valid_time  (valid_time) datetime64[us] 35kB 2015-01-01 ... 2017-12-31T18...
  * latitude    (latitude) float64 152B 79.0 76.0 73.0 70.0 ... 31.0 28.0 25.0
  * longitude   (longitude) float64 232B -25.0 -22.0 -19.0 ... 53.0 56.0 59.0
Data variables:
    2t          (valid_time, latitude, longitude) float64 19MB 247.9 ... 294.8
Attributes: (12/13)
    param:        2t
    paramId:      167
    class:        ea
    stream:       oper
    levtype:      sfc
    type:         an
    ...           ...
    date:         20150101
    time:         0
    domain:       g
    number:       0
    Conventions:  CF-1.8
    institution:  ECMWF

## Calculate the daily climatology of the ERA5 data

In [3]:
climatology_mean = ekt.climatology.mean(era5_xr)
climatology_daily_mean = ekt.climatology.daily_mean(era5_xr)
climatology_mean

<xarray.Dataset> Size: 5kB
Dimensions:    (latitude: 19, longitude: 29)
Coordinates:
  * latitude   (latitude) float64 152B 79.0 76.0 73.0 70.0 ... 31.0 28.0 25.0
  * longitude  (longitude) float64 232B -25.0 -22.0 -19.0 ... 53.0 56.0 59.0
Data variables:
    2t         (latitude, longitude) float64 4kB 255.8 259.2 ... 302.1 300.2
Attributes: (12/13)
    param:        2t
    paramId:      167
    class:        ea
    stream:       oper
    levtype:      sfc
    type:         an
    ...           ...
    date:         20150101
    time:         0
    domain:       g
    number:       0
    Conventions:  CF-1.8
    institution:  ECMWF

## Calculate the anomaly and relative anomaly

In [4]:
# The anomaly of the daily mean
anomaly_original_time_dim = ekt.climatology.anomaly(era5_xr, climatology_daily_mean)
anomaly_original_time_dim

<xarray.Dataset> Size: 19MB
Dimensions:     (valid_time: 4384, latitude: 19, longitude: 29)
Coordinates:
  * valid_time  (valid_time) datetime64[us] 35kB 2015-01-01 ... 2017-12-31T18...
    dayofyear   (valid_time) int64 35kB 1 1 1 1 2 2 ... 364 364 365 365 365 365
  * latitude    (latitude) float64 152B 79.0 76.0 73.0 70.0 ... 31.0 28.0 25.0
  * longitude   (longitude) float64 232B -25.0 -22.0 -19.0 ... 53.0 56.0 59.0
Data variables:
    2t          (valid_time, latitude, longitude) float64 19MB 2.78 ... -1.497
Attributes: (12/13)
    param:        2t
    paramId:      167
    class:        ea
    stream:       oper
    levtype:      sfc
    type:         an
    ...           ...
    date:         20150101
    time:         0
    domain:       g
    number:       0
    Conventions:  CF-1.8
    institution:  ECMWF

In [ ]:
anomaly = ekt.climatology.anomaly(era5_xr, climatology_daily_mean, frequency="dayofyear")
anomaly

<xarray.Dataset> Size: 5MB
Dimensions:     (latitude: 19, longitude: 29, valid_time: 3, dayofyear: 366)
Coordinates:
  * latitude    (latitude) float64 152B 79.0 76.0 73.0 70.0 ... 31.0 28.0 25.0
  * longitude   (longitude) float64 232B -25.0 -22.0 -19.0 ... 53.0 56.0 59.0
  * valid_time  (valid_time) datetime64[us] 24B 2015-12-31 2016-12-31 2017-12-31
  * dayofyear   (dayofyear) int64 3kB 1 2 3 4 5 6 7 ... 361 362 363 364 365 366
Data variables:
    2t          (valid_time, latitude, longitude, dayofyear) float64 5MB 10.2...
Attributes: (12/13)
    param:        2t
    paramId:      167
    class:        ea
    stream:       oper
    levtype:      sfc
    type:         an
    ...           ...
    date:         20150101
    time:         0
    domain:       g
    number:       0
    Conventions:  CF-1.8
    institution:  ECMWF

In [5]:
relative_anomaly = ekt.climatology.relative_anomaly(era5_xr, climatology_daily_mean)
relative_anomaly

<xarray.Dataset> Size: 19MB
Dimensions:     (valid_time: 4384, latitude: 19, longitude: 29)
Coordinates:
  * valid_time  (valid_time) datetime64[us] 35kB 2015-01-01 ... 2017-12-31T18...
    dayofyear   (valid_time) int64 35kB 1 1 1 1 2 2 ... 364 364 365 365 365 365
  * latitude    (latitude) float64 152B 79.0 76.0 73.0 70.0 ... 31.0 28.0 25.0
  * longitude   (longitude) float64 232B -25.0 -22.0 -19.0 ... 53.0 56.0 59.0
Data variables:
    2t          (valid_time, latitude, longitude) float64 19MB 1.134 ... -0.5052
Attributes: (12/13)
    param:        2t
    paramId:      167
    class:        ea
    stream:       oper
    levtype:      sfc
    type:         an
    ...           ...
    date:         20150101
    time:         0
    domain:       g
    number:       0
    Conventions:  CF-1.8
    institution:  ECMWF

## Plot the output for a random location

In [6]:
# Create time arrays for month and day of year data, we will base on the year 2000 but  this is arbitary and
#  hidden from plot.
latitude = 51.5
longitude = -1.0
sel_kwargs = {"latitude": latitude, "longitude": longitude, "method": "nearest"}

fig = ekp.Figure(rows=2, columns=1)

ts1 = fig.add_timeseries()
# Add the daily anomaly as a green solid line:
ts1.line(anomaly.sel(**sel_kwargs), label="Anomaly", color="seagreen")

ts2 = fig.add_timeseries()
# Add the relative anomaly as a blue dashed line:
ts2.line(relative_anomaly.sel(**sel_kwargs), units="percent", label="Relative anomaly", color="cornflowerblue")

fig.title("Daily anomaly for {variable_name}\n{location:%c}, {location:%C} ({latitude:%Lt}, {longitude:%Ln})")

ts1.ylabel()
ts2.ylabel()

fig.legend()

fig.show()

AttributeError: 'Figure' object has no attribute 'add_timeseries'

<Figure size 800x700 with 0 Axes>